# MoE: routing between experts

Top-k routing, imbalance, and total versus active parameters.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

Mixture of Experts selects a subset of experts for each token. Total size includes every expert; computation depends on active experts. Actual efficiency depends on dispatching, batch size, communication, and hardware.

This is an illustrative top-1 router. Discrete argmax does not propagate a gradient through expert selection; production MoE needs suitable weighting and balancing. This exercise studies routes, not a complete MoE trainer.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
tokens = torch.randn(32, 8)
router = nn.Linear(8, 4)
experts = nn.ModuleList([nn.Sequential(nn.Linear(8, 16), nn.GELU(), nn.Linear(16, 8)) for _ in range(4)])
probabilities = router(tokens).softmax(-1)
routes = probabilities.argmax(-1)
output = torch.zeros_like(tokens)
for index, expert in enumerate(experts):
    selected = routes == index
    if selected.any():
        output[selected] = expert(tokens[selected])
counts = torch.bincount(routes, minlength=4)
print('tokens per expert:', counts.tolist())
print('all expert parameters:', sum(p.numel() for p in experts.parameters()))
print('output shape:', output.shape)
plt.bar(range(4), counts.tolist(), color='#d73535'); plt.xlabel('expert'); plt.show()

## Try it yourself

Bias the router toward one expert and inspect usage. Why do underused experts hurt learning? What should be logged besides total loss?

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [Switch Transformers](https://arxiv.org/abs/2101.03961)